In [57]:
from recon_engine.config 

SyntaxError: invalid syntax (1192021596.py, line 1)

In [ ]:
# reconciliation.py
import pandas as pd
import json

col_names = {"Strategy":  "strategy",
             "Vehicle" :  "vehicle",
             "Date"    :   "date",
             "FIGI"    : "figi",
             "Symbol"  : "symbol",
             "Quantity" : "quantity", 
             "Market Value" : "market_value"}

sys_a = pd.read_csv('/Users/karandeepsonewane/Projects/Recon_engine/data/system_a/system_a_posns.csv')

with open('/Users/karandeepsonewane/Projects/Recon_engine/data/system_b/system_b_posns.json', 'r') as f:
    raw_data = json.load(f)
sys_b = pd.DataFrame(raw_data['data'])

with open('/Users/karandeepsonewane/Projects/Recon_engine/data/mappings/figi_cache.json', 'r') as f:
    mapping = json.load(f)
cache = pd.DataFrame(mapping.items(), columns=['figi','symbol'])

sys_a = sys_a.rename(columns=col_names)
    # type enforcement 
sys_a["date"] = pd.to_datetime(sys_a["date"])
sys_a["quantity"] = sys_a["quantity"].astype(dtype=float)
sys_a["market_value"] = sys_a["market_value"].astype(dtype=float)
print("Total Unique FIGIs", sys_a["figi"].nunique())


sys_b = sys_b.rename(columns=col_names)
sys_b["date"] = pd.to_datetime(sys_b["date"])
sys_b["quantity"] = sys_b["quantity"].astype(dtype=float)
sys_b["market_value"] = sys_b["market_value"].astype(dtype=float)

Total Unique FIGIs 132


In [44]:
print(sys_a.head(), "\n", sys_b.head(), "\n", cache.head())

   strategy vehicle       date          figi  quantity  market_value
0  Eulerian    EUGA 2025-02-05  BBG00LV100N0  -88348.0   -5232269.03
1  Eulerian    EUGA 2025-02-06  BBG00LV100N0  150165.0    8891888.65
2  Eulerian    EUGA 2025-02-07  BBG00LV100N0  150165.0    8822856.82
3  Eulerian    EUGA 2025-02-10  BBG00LV100N0  150165.0    8763681.50
4  Eulerian    EUGA 2025-02-11  BBG00LV100N0  150165.0    9056186.67 
    strategy vehicle       date symbol  quantity  market_value
0  Eulerian    EUGA 2025-02-05   BSVN  -88348.0   -5232269.03
1  Eulerian    EUGA 2025-02-06   BSVN  150165.0    8891888.65
2  Eulerian    EUGA 2025-02-07   BSVN  150165.0    8822856.82
3  Eulerian    EUGA 2025-02-10   BSVN  150165.0    8763681.50
4  Eulerian    EUGA 2025-02-11   BSVN  150165.0    9056186.67 
            figi symbol
0  BBG000B9XRY4   AAPL
1  BBG000B9YJ35    APH
2  BBG000BB72K8   LECO
3  BBG000BBJQV0   NVDA
4  BBG000BBL0Y1    ALK


In [45]:
merged_a = pd.merge(sys_a, 
                    cache, 
                    on='figi', 
                    how='left')

merged_a.head()

,strategy,vehicle,date,figi,quantity,market_value,symbol
0,Eulerian,EUGA,2025-02-05,BBG00LV100N0,-88348.0,-5232269.03,BSVN
1,Eulerian,EUGA,2025-02-06,BBG00LV100N0,150165.0,8891888.65,BSVN
2,Eulerian,EUGA,2025-02-07,BBG00LV100N0,150165.0,8822856.82,BSVN
3,Eulerian,EUGA,2025-02-10,BBG00LV100N0,150165.0,8763681.50,BSVN
4,Eulerian,EUGA,2025-02-11,BBG00LV100N0,150165.0,9056186.67,BSVN


In [46]:
all_merged = pd.merge(merged_a, 
                    sys_b, 
                    on=['strategy', 'vehicle', 'date', 'symbol'], 
                    how='outer', 
                    suffixes=("_a", "_b"),
                    indicator=True)

all_merged

,strategy,vehicle,date,figi,quantity_a,market_value_a,symbol,quantity_b,market_value_b,_merge
0,Collatz,COMF,2025-02-05,BBG005P7Q881,159255.0,4.732180e+07,AAL,159255.0,47321796.12,both
1,Collatz,COMF,2025-02-05,BBG000B9XRY4,198695.0,1.919185e+07,AAPL,198695.0,19191846.79,both
2,Collatz,COMF,2025-02-05,BBG000B9YJ35,-119904.0,-9.573193e+06,APH,-119904.0,-9573192.79,both
3,Collatz,COMF,2025-02-05,BBG000BCTLF6,111585.0,4.898527e+07,BAC,108028.0,48985272.60,both
4,Collatz,COMF,2025-02-05,BBG000BKD3K9,194602.0,2.599473e+07,BLDR,194602.0,25994725.06,both
...,...,...,...,...,...,...,...,...,...,...
97531,Hamiltonian,HASYY,2026-03-20,BBG005NTTSP9,46745.0,1.225263e+07,SG,46745.0,12252631.89,both
97532,Hamiltonian,HASYY,2026-03-20,BBG000BT1715,141769.0,2.838892e+07,SJM,141769.0,28388915.68,both
97533,Hamiltonian,HASYY,2026-03-20,BBG000D2JB50,-120852.0,-2.063671e+07,SKT,-120852.0,-20636706.86,both
97534,Hamiltonian,HASYY,2026-03-20,BBG000BL79J3,59878.0,2.100117e+06,URBN,59878.0,2100117.00,both


In [56]:
# Cell: Compute diffs
all_merged["qty_diff"] = all_merged["quantity_a"].fillna(0) - all_merged["quantity_b"].fillna(0)
all_merged["mv_diff"] = all_merged["market_value_a"].fillna(0) - all_merged["market_value_b"].fillna(0)

In [ ]:


# Break classification
def classify(row):
    # for orphan records
    if row["_merge"] == "left_only": 
        return "system_a_only"
    if row["_merge"] == "right_only":
        return "system_b_only"

    qty_ok = row["qty_diff"] == 0
    mv_ok = abs(row["mv_diff"]) <= 0.01 # with tolerance -- add it to config later on
    
    # recon break
    if qty_ok and mv_ok:
        return "match"
    if not qty_ok and not mv_ok:
        return "both_break"
    if not qty_ok:
        return "qty_break"
    return "mv_break"

all_merged["status"] = all_merged.apply(classify, axis=1)
print(all_merged["status"].value_counts())

status
match            96033
qty_break          764
mv_break           738
system_a_only        1
Name: count, dtype: int64


In [54]:
# Cell: Inspect breaks
breaks = all_merged[all_merged["status"].isin(["qty_break", "mv_break", "both_break"])]
print(f"Total breaks: {len(breaks)}")
breaks[["strategy","vehicle","date","symbol","quantity_a","quantity_b","qty_diff","market_value_a","market_value_b","mv_diff","status"]].head(10)

Total breaks: 1502


,strategy,vehicle,date,symbol,quantity_a,quantity_b,qty_diff,market_value_a,market_value_b,mv_diff,status
3,Collatz,COMF,2025-02-05,BAC,111585.0,108028.0,3557.0,48985272.60,48985272.60,0.00,qty_break
11,Collatz,COMF,2025-02-05,KRE,70878.0,73609.0,-2731.0,22616887.03,22616887.03,0.00,qty_break
16,Collatz,COMF,2025-02-05,NET,8837.0,8932.0,-95.0,1441088.19,1441088.19,0.00,qty_break
32,Collatz,COMF,2025-02-05,VSCO,189900.0,197093.0,-7193.0,60839619.16,60839619.16,0.00,qty_break
90,Collatz,COMF,2025-02-07,PHR,151011.0,151011.0,0.0,56713688.88,56682258.77,31430.11,mv_break
100,Collatz,COMF,2025-02-07,TJX,-164354.0,-164354.0,0.0,-3436745.62,-3429342.04,-7403.58,mv_break
157,Collatz,COMF,2025-02-11,MMM,-80037.0,-80037.0,0.0,-25967755.00,-25988431.14,20676.14,mv_break
268,Collatz,COMF,2025-02-14,REGN,-73657.0,-74396.0,739.0,-9986474.79,-9986474.79,0.00,qty_break
280,Collatz,COMF,2025-02-14,XOP,-163092.0,-161784.0,-1308.0,-29022215.62,-29022215.62,0.00,qty_break
574,Collatz,COMF,2025-02-28,LECO,-156316.0,-154255.0,-2061.0,-45628430.93,-45628430.93,0.00,qty_break


In [55]:
# Cell: Validate known good data — BSVN should match exactly
bsvn = all_merged[all_merged["symbol"] == "BSVN"].head(5)
print(bsvn[["date","symbol","quantity_a","quantity_b","qty_diff","mv_diff","status"]])

            date symbol  quantity_a  quantity_b  qty_diff  mv_diff status
13422 2025-02-05   BSVN    -88348.0    -88348.0       0.0      0.0  match
13465 2025-02-06   BSVN    150165.0    150165.0       0.0      0.0  match
13506 2025-02-07   BSVN    150165.0    150165.0       0.0      0.0  match
13546 2025-02-10   BSVN    150165.0    150165.0       0.0      0.0  match
13587 2025-02-11   BSVN    150165.0    150165.0       0.0      0.0  match


In [ ]:
# for strategy 
# first aggregate across vehicle on strategy level and then join using strategy, date and symbol

merged 